# Strategy:
1. Long: enter at below 35 rsi, exit at above 70
2. Short: enter at above 70, exit at 80% of entry RSI`

In [1]:
from utilities import *

df = load_file("./Data/SPY_FULL.csv", date_format="%d/%m/%Y")
df = add_rsi(df)
print(df.dtypes)
print(df.head())
print(df.tail())

timestamp    datetime64[ns]
open                float64
high                float64
low                 float64
close               float64
volume                int64
rsi                 float64
dtype: object
   timestamp      open      high      low     close   volume  rsi
0 1999-11-19  142.4062  142.9687  142.000  142.5000  4832100  NaN
1 1999-11-22  142.4375  143.0000  141.500  142.4687  4155400  NaN
2 1999-11-23  142.8437  142.8437  140.375  141.2187  5918000  NaN
3 1999-11-24  140.7500  142.4375  140.000  141.9687  4459700  NaN
4 1999-11-26  142.4687  142.8750  141.250  141.4375  1693900  NaN
      timestamp    open    high       low   close    volume        rsi
6352 2025-02-24  602.02  603.03  596.4900  597.21  50737213  44.002908
6353 2025-02-25  597.15  597.89  589.5600  594.24  58266472  41.409659
6354 2025-02-26  595.93  599.58  591.8556  594.54  43321578  41.782877
6355 2025-02-27  596.85  598.02  584.6500  585.05  74196664  34.332585
6356 2025-02-28  585.56  594.72  582.44

In [ ]:
stats = {
    "profit_loss_log": [],
    "total_trades": 0,
    "winning_trades": 0,
    "long_array_idx": [],
    "short_array_idx": []
}
verbose_log = ("Trade Log\n" + "="*40 + "\n")

open_positions = []
for i, row in df.iterrows():
    ### DEFINE WHAT YOU NEED HERE ###
    rsi = row["rsi"]

    ### ENTRY ###
    if rsi < 35:
        ### ENTRY LONG CONDITIONS ###
        open_positions.append(
            {"type": "LONG", "entry": row["close"], "entry_date": row["timestamp"], "entry_rsi": rsi}
        )
        stats["long_array_idx"].append(i)
    elif rsi > 70:
        ### ENTRY SHORT CONDITIONS ###
        open_positions.append(
            {"type": "SHORT", "entry": row["close"], "entry_date": row["timestamp"], "entry_rsi": rsi}
        )
        stats["short_array_idx"].append(i)

    ### EXIT ###
    for idx in reversed(range(len(open_positions))): # we go backwards so we can safely pop elements
        position = open_positions[idx]

        if position["type"] == "LONG":
            ### EXIT LONG CONDITIONS ###
            if rsi >= 80:
                stats, log = close_trade(position, row, stats)
                verbose_log += log
                open_positions.pop(idx)

        elif position["type"] == "SHORT":
            ### EXIT SHORT CONDITIONS ###
            if rsi <= 0.8 * position['entry_rsi']:
                stats, log = close_trade(position, row, stats)
                verbose_log += log
                open_positions.pop(idx)

# Calculate and log statistics
win_rate = (stats["winning_trades"] / stats["total_trades"]) * 100 if stats["total_trades"] > 0 else 0

verbose_log += ("\nSummary Statistics\n")
verbose_log += ("=" * 40 + "\n")
verbose_log += (f"Total Trades: {stats["total_trades"]}\n")
verbose_log += (f"Win Rate: {win_rate:.2f}%\n")
verbose_log += (f"Total Profit/Loss: {sum(stats["profit_loss_log"]):.2f}%\n")

write_to_file("trade_log.txt", verbose_log)

# Console outputs
print(f"Total Trades: {stats["total_trades"]}")
print(f"Win Rate: {win_rate:.2f}%")
print(f"Total Profit/Loss: {sum(stats["profit_loss_log"]):.2f}%")
print(f"Buy and Hold Return: {percent_change(df['close'].iloc[0], df['close'].iloc[-1])}%")

Total Trades: 733
Win Rate: 78.31%
Total Profit/Loss: 19602.74%
Underlying Growth: 316.97%


rough idea: use past longer period standard dev to identify if market is sideways, then use RSI to entry exit

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, 
                    subplot_titles=("Stock Price", "RSI"))

fig.add_trace(
    go.Scatter(x=df['timestamp'], y=df['close'], mode='lines', name='Stock Price', line=dict(color='blue')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=df['timestamp'], y=df['rsi'], mode='lines', name='RSI', line=dict(color='red')),
    row=2, col=1
)

fig.add_shape(type="line", x0=df['timestamp'].min(), x1=df['timestamp'].max(), y0=35, y1=35,
              line=dict(color="grey", width=2, dash="dot"), row=2, col=1)

fig.add_shape(type="line", x0=df['timestamp'].min(), x1=df['timestamp'].max(), y0=70, y1=70,
              line=dict(color="grey", width=2, dash="dot"), row=2, col=1)

for idx in stats["long_array_idx"]:
    fig.add_annotation(dict(
        x=df['timestamp'][idx],
        y=df['close'][idx],
        xref="x1", yref="y1",
        text="⬆",
        showarrow=False,
        font=dict(color="#03fc17", size=10)
    ))

for idx in stats["short_array_idx"]:
    fig.add_annotation(dict(
        x=df['timestamp'][idx],
        y=df['close'][idx],
        xref="x1", yref="y1",
        text="⬇",
        showarrow=False,
        font=dict(color="#fc0303", size=10)
    ))

fig.update_layout(height=600, title="Stock Price and RSI", showlegend=False)
fig.show()